# Загрузка данных Brandpulse API в DuckDB #

In [ ]:
#Если не установлен модуль duckdb, то в ячейке необходимо выполнить "!pip install duckdb"
!pip install duckdb

In [ ]:
import os
import glob
from brandpulse_api.data import BrandpulseData
from brandpulse_api.duckdb import DuckDbData

Задаем параметры

In [ ]:
# параметры сохранения даннных
DOWNLOADS_DIR = 'packed_data'
EXTRACT_DIR = 'unpacked_data'
DB_FILE = 'brandpulse3.duckdb'

# пути для загрузки данных
DICTIONARY_DATA = f'{EXTRACT_DIR}/dictionary'
ANSWER_DATA = f'{EXTRACT_DIR}/data/answer'
WEIGHT_DATA = f'{EXTRACT_DIR}/data/weight'
PROFILE_ANSWER_DATA = f'{EXTRACT_DIR}/data/profile/answer'
PROFILE_WEIGHT_DATA = f'{EXTRACT_DIR}/data/profile/weight'
BOOST_ANSWER_DATA = f'{EXTRACT_DIR}/data/profile/boost/answer'


In [ ]:
# создаем объекты для работы
bp = BrandpulseData(download_dir=DOWNLOADS_DIR, extract_dir=EXTRACT_DIR)

db = DuckDbData(DB_FILE)

Загружаем список доступных проектов 

In [ ]:
# получить список доступных проектов
df_projects = bp.get_available_projects()

# сохраняем доступные проекты
os.makedirs(EXTRACT_DIR, exist_ok=True)
filepath = os.path.join(EXTRACT_DIR, f'projects.csv.gz')
df_projects.to_csv(filepath, compression='gzip', index=False)
print(f'Data successfully downloaded to {filepath}')

# загрузка доступных проектов
db.load_all_data_from_file(table_name="projects", file_path="unpacked_data/projects.csv.gz")

Загружаем данные справочников

In [ ]:
# получить данные справочников
bp.get_dictionary()

# загрузка словарей
for fpath in glob.glob(os.path.join(DICTIONARY_DATA, '*.csv.gz')):
    db.load_all_data_from_file(table_name=os.path.basename(fpath).split('.')[0], file_path=fpath)

Загружаем данные профиля

In [ ]:
# получить данные профиля (важно: на загрузку профиля может уходить до 5 минут)
bp.get_profile(2024)

# загрузка ответов для профиля
for fpath in glob.glob(os.path.join(PROFILE_ANSWER_DATA, '*.csv.gz')):
    db.load_project_data_from_file(table_name="answer", file_path=fpath)

# загрузка весов для профиля
for fpath in glob.glob(os.path.join(PROFILE_WEIGHT_DATA, '*.csv.gz')):
    db.load_project_data_from_file(table_name="weight", file_path=fpath)

Загрузка данных проектов

In [ ]:
project_list = ['3468876519813574080'] #Даже для работы только с Профилем необходимо иметь загруженным 1 проект BHT

In [ ]:
# получить данные проектов 
bp.get_projects(project_ids=project_list)

# загрузка ответов для проектов
for fpath in glob.glob(os.path.join(ANSWER_DATA, '*.csv.gz')):
    db.load_project_data_from_file(table_name="answer", file_path=fpath)
    
# загрузка весов для проектов
for fpath in glob.glob(os.path.join(WEIGHT_DATA, '*.csv.gz')):
    db.load_project_data_from_file(table_name="weight", file_path=fpath)

Загрузка бустовых данных проектов

In [ ]:
# получить бустовые данные проектов
bp.get_boost_answer(project_ids=project_list)

# загрузка бустовых ответов для проектов
for fpath in glob.glob(os.path.join(BOOST_ANSWER_DATA, '*.csv.gz')):
    # достаем номер проекта 
    db.load_project_data_from_file(table_name="boost_answer", file_path=fpath)

Очистка данные проектов (удаление ответов и весов)

In [ ]:
#bp.delete_table_project_data(['8079710695247165824','4981228102163853632','513659322260867072'])